# mesh4.msh

In [1]:
import gmsh

gmsh.initialize()
gmsh.model.add("mesh4")

# gmsh.option → settings
# gmsh.model → current geometric model you are building (volumes, physical groups, mesh)
# gmsh.model.occ → geometry operations (boxes, cylinders, fragment, com)
# gmsh.fltk → graphical interface tools

### Geometry parameters

In [2]:
# length in x,y
Lx = 10.0
Ly = 10.0

# height in z (elevation)

z1 = 5.0        #<---- surface
z2 = 3.5        
z3 = 3.0        
z4 = 2.0        
z5 = 1.5        
z0 = 0.0        #<---- bottom

# Layer 1 (surface): z2 to z1
# Layer 2 (cap rock): z3 to z2
# Layer 3 (reservoir): z4 to z3   <-- layer of interest
# Layer 4 (cap rock): z5 to z4
# Layer 5 (bottom): z0 to z5

# layer 3 = Middle point for well completition
z_middle_central = (z4 + z3) / 2.0   #----------2.5

# Well radius
rw = 0.2

# Well positions (in opposite corners)

xw1, yw1 = 1.0, 1.0
xw2, yw2 = 9.0, 9.0


### Layered boxes (addBox)

In [3]:
# Creating 5 layers = 5 stacked boxes
# gmsh.model.occ.addBox(x, y, z, dx, dy, dz)
# Starting corner point (x, y, z)
# extend it by (dx, dy, dz)

layer1 = gmsh.model.occ.addBox(0, 0, z2, Lx, Ly, z1 - z2)
layer2 = gmsh.model.occ.addBox(0, 0, z3, Lx, Ly, z2 - z3)
layer3 = gmsh.model.occ.addBox(0, 0, z4, Lx, Ly, z3 - z4)
layer4 = gmsh.model.occ.addBox(0, 0, z5, Lx, Ly, z4 - z5)
layer5 = gmsh.model.occ.addBox(0, 0, z0, Lx, Ly, z5 - z0)

# Layer 1 (surface): z2 to z1
# Layer 2 (cap rock): z3 to z2
# Layer 3 (reservoir): z4 to z3   <-- layer of interest
# Layer 4 (cap rock): z5 to z4
# Layer 5 (bottom): z0 to z5

# tag = entity’s ID assigned by Gmsh (dim, tag)

for tag in [layer1, layer2, layer3, layer4, layer5]:
    com = gmsh.model.occ.getCenterOfMass(3, tag)
    print(f"Layer {tag} center of mass = {com}")

Layer 1 center of mass = (5.0, 5.0, 4.25)
Layer 2 center of mass = (5.0, 5.0, 3.25)
Layer 3 center of mass = (5.0, 5.0, 2.5)
Layer 4 center of mass = (5.0, 5.0, 1.7500000000000002)
Layer 5 center of mass = (5.0, 5.0, 0.75)


### Wells (addCylinder)

In [4]:

well_length = z1 - z_middle_central   # From top surface (z1) down to middle of central layer (z_middle_central)

#addCylinder(x, y, z, dx, dy, dz, r)

well1 = gmsh.model.occ.addCylinder(xw1, yw1, z1, 0, 0, -well_length, rw)
well2 = gmsh.model.occ.addCylinder(xw2, yw2, z1, 0, 0, -well_length, rw)


com1 = gmsh.model.occ.getCenterOfMass(3, well1)
com2 = gmsh.model.occ.getCenterOfMass(3, well2)

print(f"Well 1 → tag = {well1}, center of mass = {com1}")
print(f"Well 2 → tag = {well2}, center of mass = {com2}")
print(well_length, z_middle_central)

Well 1 → tag = 6, center of mass = (1.0, 1.0, 3.75)
Well 2 → tag = 7, center of mass = (8.999999999999998, 8.999999999999998, 3.75)
2.5 2.5


## Fragment
### explanation***********

In [5]:
# (dimension, entity tags)
domain_volumes = [(3, layer1), (3, layer2), (3, layer3), (3, layer4), (3, layer5)]
well_volumes = [(3, well1), (3, well2)]


# fragment → boolean partitioning operation, splits complex geometries or networks into subsets using boolean logic (AND, OR, NOT) to 
#            simplify, simulate, or analyze subcomponents, such as separating combined 3D shapes (union/subtraction) or decomposing Boolean networks. 

gmsh.model.occ.fragment(domain_volumes, well_volumes)

gmsh.model.occ.synchronize()

In [6]:
for dim, tag in gmsh.model.getEntities(3):
    x, y, z = gmsh.model.occ.getCenterOfMass(dim, tag)
    
    print(f"Volume {tag} | COM = ({x:.2f}, {y:.2f}, {z:.2f})")

Volume 4 | COM = (5.00, 5.00, 1.75)
Volume 5 | COM = (5.00, 5.00, 0.75)
Volume 6 | COM = (5.00, 5.00, 4.25)
Volume 7 | COM = (1.00, 1.00, 4.25)
Volume 8 | COM = (9.00, 9.00, 4.25)
Volume 9 | COM = (5.00, 5.00, 3.25)
Volume 10 | COM = (1.00, 1.00, 3.25)
Volume 11 | COM = (9.00, 9.00, 3.25)
Volume 12 | COM = (5.00, 5.00, 2.50)
Volume 13 | COM = (1.00, 1.00, 2.75)
Volume 14 | COM = (9.00, 9.00, 2.75)


In [8]:
for dim, tag in gmsh.model.getEntities(2):
    x, y, z = gmsh.model.occ.getCenterOfMass(dim, tag)
    
    print(f"Surface {tag} | COM = ({x:.2f}, {y:.2f}, {z:.2f})")

Surface 1 | COM = (0.00, 5.00, 4.25)
Surface 2 | COM = (5.00, 0.00, 4.25)
Surface 3 | COM = (5.00, 5.00, 5.00)
Surface 4 | COM = (5.00, 10.00, 4.25)
Surface 5 | COM = (5.00, 5.00, 3.50)
Surface 6 | COM = (10.00, 5.00, 4.25)
Surface 7 | COM = (1.00, 1.00, 4.25)
Surface 8 | COM = (9.00, 9.00, 4.25)
Surface 9 | COM = (1.00, 1.00, 5.00)
Surface 10 | COM = (1.00, 1.00, 3.50)
Surface 11 | COM = (9.00, 9.00, 5.00)
Surface 12 | COM = (9.00, 9.00, 3.50)
Surface 13 | COM = (0.00, 5.00, 3.25)
Surface 14 | COM = (5.00, 0.00, 3.25)
Surface 15 | COM = (5.00, 10.00, 3.25)
Surface 16 | COM = (5.00, 5.00, 3.00)
Surface 17 | COM = (10.00, 5.00, 3.25)
Surface 18 | COM = (1.00, 1.00, 3.25)
Surface 19 | COM = (9.00, 9.00, 3.25)
Surface 20 | COM = (1.00, 1.00, 3.00)
Surface 21 | COM = (9.00, 9.00, 3.00)
Surface 22 | COM = (0.00, 5.00, 2.50)
Surface 23 | COM = (5.00, 0.00, 2.50)
Surface 24 | COM = (5.00, 10.00, 2.50)
Surface 25 | COM = (5.00, 5.00, 2.00)
Surface 26 | COM = (10.00, 5.00, 2.50)
Surface 27 | CO

### Physical Groups

In [9]:
# -----------------------------
# ROCK / DOMAIN VOLUMES
# -----------------------------
gmsh.model.addPhysicalGroup(3, [6], 105)
gmsh.model.setPhysicalName(3, 105, "Layer1_Top")

gmsh.model.addPhysicalGroup(3, [9], 104)
gmsh.model.setPhysicalName(3, 104, "Layer2_caprock1")

gmsh.model.addPhysicalGroup(3, [12], 103)
gmsh.model.setPhysicalName(3, 103, "Layer3_Reservoir")

gmsh.model.addPhysicalGroup(3, [4], 102)
gmsh.model.setPhysicalName(3, 102, "Layer4_caprock2")

gmsh.model.addPhysicalGroup(3, [5], 101)
gmsh.model.setPhysicalName(3, 101, "Layer5_bottom")


# -----------------------------
# WELL 1 VOLUMES
# -----------------------------
gmsh.model.addPhysicalGroup(3, [13, 10, 7], 201)
gmsh.model.setPhysicalName(3, 201, "Well1")

# -----------------------------
# WELL 2 VOLUMES
# -----------------------------
gmsh.model.addPhysicalGroup(3, [14, 11, 8], 202)
gmsh.model.setPhysicalName(3, 202, "Well2")



for dim, tag in gmsh.model.getPhysicalGroups():
    print(dim, tag, gmsh.model.getPhysicalName(dim, tag))

3 101 Layer5_bottom
3 102 Layer4_caprock2
3 103 Layer3_Reservoir
3 104 Layer2_caprock1
3 105 Layer1_Top
3 201 Well1
3 202 Well2


In [10]:
# WELL 1 SURFACES

for v in [7, 10, 13]:
    print("Well1 Volume", v)
    for dim, tag in gmsh.model.getBoundary([(3, v)], oriented=False, recursive=False):
        if dim == 2:
            com = gmsh.model.occ.getCenterOfMass(2, tag)
            print("  Surface", tag, "com =", com)

Well1 Volume 7
  Surface 7 com = (1.0, 1.0, 4.249999999999999)
  Surface 9 com = (1.0, 1.0, 5.0)
  Surface 10 com = (1.0, 1.0, 3.5)
Well1 Volume 10
  Surface 10 com = (1.0, 1.0, 3.5)
  Surface 18 com = (1.0, 1.0, 3.25)
  Surface 20 com = (1.0, 1.0, 3.0)
Well1 Volume 13
  Surface 20 com = (1.0, 1.0, 3.0)
  Surface 27 com = (1.0, 1.0, 2.75)
  Surface 29 com = (1.0, 1.0, 2.5)


In [11]:
# WELL 2 SURFACES

for v in [8, 11, 14]:
    print("Well2 Volume", v)
    for dim, tag in gmsh.model.getBoundary([(3, v)], oriented=False, recursive=False):
        if dim == 2:
            com = gmsh.model.occ.getCenterOfMass(2, tag)
            print("  Surface", tag, "com =", com)

Well2 Volume 8
  Surface 8 com = (9.0, 9.0, 4.249999999999999)
  Surface 11 com = (9.0, 9.0, 5.0)
  Surface 12 com = (9.0, 9.0, 3.5)
Well2 Volume 11
  Surface 12 com = (9.0, 9.0, 3.5)
  Surface 19 com = (9.0, 9.0, 3.25)
  Surface 21 com = (9.0, 9.0, 3.0)
Well2 Volume 14
  Surface 21 com = (9.0, 9.0, 3.0)
  Surface 28 com = (9.0, 9.0, 2.75)
  Surface 30 com = (9.0, 9.0, 2.5)


In [12]:
# --------------------------------------------------
# Create physical groups for well surfaces:
# --------------------------------------------------

# -----------------------------
# WELL 1 SURFACES
# -----------------------------
gmsh.model.addPhysicalGroup(2, [9], 401)
gmsh.model.setPhysicalName(2, 401, "Well1_Top")

gmsh.model.addPhysicalGroup(2, [27], 402)
gmsh.model.setPhysicalName(2, 402, "Well1_compl")

# -----------------------------
# WELL 2 SURFACES
# -----------------------------
gmsh.model.addPhysicalGroup(2, [11], 403)
gmsh.model.setPhysicalName(2, 403, "Well2_Top")

gmsh.model.addPhysicalGroup(2, [28], 404)
gmsh.model.setPhysicalName(2, 404, "Well2_compl")

for dim, tag in gmsh.model.getPhysicalGroups(2):
    print(dim, tag, gmsh.model.getPhysicalName(dim, tag))


2 401 Well1_Top
2 402 Well1_compl
2 403 Well2_Top
2 404 Well2_compl


In [13]:
# --------------------------------------------------
# Create physical groups for the formation (outer boundaries of the box)
# --------------------------------------------------

# Left boundary x = 0
gmsh.model.addPhysicalGroup(2, [1, 13, 22, 31, 36], 301)
gmsh.model.setPhysicalName(2, 301, "Xmin")

# Right boundary x = 10
gmsh.model.addPhysicalGroup(2, [6, 17, 26, 32, 37], 302)
gmsh.model.setPhysicalName(2, 302, "Xmax")

# Front boundary y = 0
gmsh.model.addPhysicalGroup(2, [2, 14, 23, 33, 38], 303)
gmsh.model.setPhysicalName(2, 303, "Ymin")

# Back boundary y = 10
gmsh.model.addPhysicalGroup(2, [4, 15, 24, 34, 39], 304)
gmsh.model.setPhysicalName(2, 304, "Ymax")

# Top boundary z = 5
gmsh.model.addPhysicalGroup(2, [3], 305)
gmsh.model.setPhysicalName(2, 305, "Zmax")

# Bottom boundary z = 0
gmsh.model.addPhysicalGroup(2, [40], 306)
gmsh.model.setPhysicalName(2, 306, "Zmin")

for dim, tag in gmsh.model.getPhysicalGroups(2):
    print(dim, tag, gmsh.model.getPhysicalName(dim, tag))

2 301 Xmin
2 302 Xmax
2 303 Ymin
2 304 Ymax
2 305 Zmax
2 306 Zmin
2 401 Well1_Top
2 402 Well1_compl
2 403 Well2_Top
2 404 Well2_compl


In [14]:

# --------------------------------------------------
# 5. GLOBAL MESH SIZES
#    Assign a general size to all points
# --------------------------------------------------
all_points = gmsh.model.getEntities(0)
gmsh.model.mesh.setSize(all_points, 2)


In [15]:

# --------------------------------------------------
# 6. REFINEMENT BY LAYER USING BOX FIELDS
#    Small size = more refined
# --------------------------------------------------
# Layer 1 and 5: coarsest
field1 = gmsh.model.mesh.field.add("Box")
gmsh.model.mesh.field.setNumber(field1, "VIn", 2)
gmsh.model.mesh.field.setNumber(field1, "VOut", 2)
gmsh.model.mesh.field.setNumber(field1, "XMin", 0)
gmsh.model.mesh.field.setNumber(field1, "XMax", Lx)
gmsh.model.mesh.field.setNumber(field1, "YMin", 0)
gmsh.model.mesh.field.setNumber(field1, "YMax", Ly)
gmsh.model.mesh.field.setNumber(field1, "ZMin", z2)
gmsh.model.mesh.field.setNumber(field1, "ZMax", z1)

field5 = gmsh.model.mesh.field.add("Box")
gmsh.model.mesh.field.setNumber(field5, "VIn", 2)
gmsh.model.mesh.field.setNumber(field5, "VOut", 2)
gmsh.model.mesh.field.setNumber(field5, "XMin", 0)
gmsh.model.mesh.field.setNumber(field5, "XMax", Lx)
gmsh.model.mesh.field.setNumber(field5, "YMin", 0)
gmsh.model.mesh.field.setNumber(field5, "YMax", Ly)
gmsh.model.mesh.field.setNumber(field5, "ZMin", z0)
gmsh.model.mesh.field.setNumber(field5, "ZMax", z5)


In [16]:

# Layer 3: most refined
field3 = gmsh.model.mesh.field.add("Box")
gmsh.model.mesh.field.setNumber(field3, "VIn", 0.2)
gmsh.model.mesh.field.setNumber(field3, "VOut", 1)
gmsh.model.mesh.field.setNumber(field3, "XMin", 0)
gmsh.model.mesh.field.setNumber(field3, "XMax", Lx)
gmsh.model.mesh.field.setNumber(field3, "YMin", 0)
gmsh.model.mesh.field.setNumber(field3, "YMax", Ly)
gmsh.model.mesh.field.setNumber(field3, "ZMin", z4)
gmsh.model.mesh.field.setNumber(field3, "ZMax", z3)


In [17]:

# Layer 2 and 4 - cap rocks: intermediate
field2 = gmsh.model.mesh.field.add("Box")
gmsh.model.mesh.field.setNumber(field2, "VIn", 1)
gmsh.model.mesh.field.setNumber(field2, "VOut", 1)
gmsh.model.mesh.field.setNumber(field2, "XMin", 0)
gmsh.model.mesh.field.setNumber(field2, "XMax", Lx)
gmsh.model.mesh.field.setNumber(field2, "YMin", 0)
gmsh.model.mesh.field.setNumber(field2, "YMax", Ly)
gmsh.model.mesh.field.setNumber(field2, "ZMin", z3)
gmsh.model.mesh.field.setNumber(field2, "ZMax", z2)

field4 = gmsh.model.mesh.field.add("Box")
gmsh.model.mesh.field.setNumber(field4, "VIn", 1)
gmsh.model.mesh.field.setNumber(field4, "VOut", 1)
gmsh.model.mesh.field.setNumber(field4, "XMin", 0)
gmsh.model.mesh.field.setNumber(field4, "XMax", Lx)
gmsh.model.mesh.field.setNumber(field4, "YMin", 0)
gmsh.model.mesh.field.setNumber(field4, "YMax", Ly)
gmsh.model.mesh.field.setNumber(field4, "ZMin", z5)
gmsh.model.mesh.field.setNumber(field4, "ZMax", z4)


In [18]:

# --------------------------------------------------
# 7. EXTRA REFINEMENT AROUND WELLS
# --------------------------------------------------
well_field1 = gmsh.model.mesh.field.add("Cylinder")
gmsh.model.mesh.field.setNumber(well_field1, "VIn", 0.3)
gmsh.model.mesh.field.setNumber(well_field1, "VOut", 0.3)
gmsh.model.mesh.field.setNumber(well_field1, "XCenter", xw1)
gmsh.model.mesh.field.setNumber(well_field1, "YCenter", yw1)
gmsh.model.mesh.field.setNumber(well_field1, "ZCenter", z_middle_central)
gmsh.model.mesh.field.setNumber(well_field1, "XAxis", 0)
gmsh.model.mesh.field.setNumber(well_field1, "YAxis", 0)
gmsh.model.mesh.field.setNumber(well_field1, "ZAxis", z1)
gmsh.model.mesh.field.setNumber(well_field1, "Radius", 1.5)

well_field2 = gmsh.model.mesh.field.add("Cylinder")
gmsh.model.mesh.field.setNumber(well_field2, "VIn", 0.3)
gmsh.model.mesh.field.setNumber(well_field2, "VOut", 0.3)
gmsh.model.mesh.field.setNumber(well_field2, "XCenter", xw2)
gmsh.model.mesh.field.setNumber(well_field2, "YCenter", yw2)
gmsh.model.mesh.field.setNumber(well_field2, "ZCenter", z_middle_central)
gmsh.model.mesh.field.setNumber(well_field2, "XAxis", 0)
gmsh.model.mesh.field.setNumber(well_field2, "YAxis", 0)
gmsh.model.mesh.field.setNumber(well_field2, "ZAxis", z1)
gmsh.model.mesh.field.setNumber(well_field2, "Radius", 1.5)


In [19]:

# --------------------------------------------------
# 8. COMBINE FIELDS
#    Use the minimum size from all refinement fields
# --------------------------------------------------
min_field = gmsh.model.mesh.field.add("Min")
gmsh.model.mesh.field.setNumbers(
    min_field,
    "FieldsList",
    [field1, field2, field3, field4, field5, well_field1, well_field2]
)
gmsh.model.mesh.field.setAsBackgroundMesh(min_field)

# Optionals: No point-based or curvature-based automatic sizing
# Mesh.MeshSizeExtendFromBoundary = 0, avoid automatically extend boundary-based sizes into the interior
# Mesh.MeshSizeFromPoints = 0, Gmsh ignores point-based sizing as a primary source.
# Mesh.MeshSizeFromCurvature = 0, not want automatic curvature-based refinement interfering.
gmsh.option.setNumber("Mesh.MeshSizeExtendFromBoundary", 0)
gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 0)


In [20]:
# --------------------------------------------------
# 9. GENERATE 3D MESH
# --------------------------------------------------
gmsh.model.mesh.generate(3)


In [21]:
for v in gmsh.model.getEntities(3):
    elem_types, elem_tags, elem_node_tags = gmsh.model.mesh.getElements(3, v[1])
    n_elems = sum(len(tags) for tags in elem_tags)
    print(f"Volume {v[1]} has {n_elems} elements")

Volume 4 has 19240 elements
Volume 5 has 27141 elements
Volume 6 has 27214 elements
Volume 7 has 81 elements
Volume 8 has 75 elements
Volume 9 has 18912 elements
Volume 10 has 47 elements
Volume 11 has 44 elements
Volume 12 has 60386 elements
Volume 13 has 66 elements
Volume 14 has 68 elements


In [22]:
# --------------------------------------------------
# 10. SAVE
# --------------------------------------------------
gmsh.write("mesh4.msh")


### Checking physical tags in the mesh

In [23]:
import meshio
import numpy as np

mesh = meshio.read("mesh4.msh")

for i, cell_block in enumerate(mesh.cells):
    vals = mesh.cell_data["gmsh:physical"][i]
    print(f"Block {i}: type = {cell_block.type}, tags = {np.unique(vals)}")


Block 0: type = triangle, tags = [301]
Block 1: type = triangle, tags = [303]
Block 2: type = triangle, tags = [305]
Block 3: type = triangle, tags = [304]
Block 4: type = triangle, tags = [302]
Block 5: type = triangle, tags = [401]
Block 6: type = triangle, tags = [403]
Block 7: type = triangle, tags = [301]
Block 8: type = triangle, tags = [303]
Block 9: type = triangle, tags = [304]
Block 10: type = triangle, tags = [302]
Block 11: type = triangle, tags = [301]
Block 12: type = triangle, tags = [303]
Block 13: type = triangle, tags = [304]
Block 14: type = triangle, tags = [302]
Block 15: type = triangle, tags = [402]
Block 16: type = triangle, tags = [404]
Block 17: type = triangle, tags = [301]
Block 18: type = triangle, tags = [302]
Block 19: type = triangle, tags = [303]
Block 20: type = triangle, tags = [304]
Block 21: type = triangle, tags = [301]
Block 22: type = triangle, tags = [302]
Block 23: type = triangle, tags = [303]
Block 24: type = triangle, tags = [304]
Block 25:

In [24]:
import meshio
import numpy as np

mesh = meshio.read("mesh4.msh")

cells_out = []
phys_out = []
geom_out = []

for i, cell_block in enumerate(mesh.cells):
    cell_type = cell_block.type
    
    if cell_type in ["tetra", "triangle"]:
        cells_out.append((cell_type, cell_block.data))
        phys_out.append(mesh.cell_data["gmsh:physical"][i])
        geom_out.append(mesh.cell_data["gmsh:geometrical"][i])

new_mesh = meshio.Mesh(
    points=mesh.points,
    cells=cells_out,
    cell_data={
        "gmsh:physical": phys_out,
        "gmsh:geometrical": geom_out,
    },
)

meshio.write("mesh4.vtu", new_mesh)

In [ ]:

# Optional visualization
gmsh.fltk.run()

gmsh.finalize()